## 1. Download Games


In [ ]:
import pandas as pd

from huggingface_hub import login
from datasets import load_dataset

import chess
import io

In [ ]:

# Huggingface login with redacted token
login("REDACTED")

In [182]:

# Load a small fragment of Lichess dataset
ds = load_dataset(
    "parquet", split="train", streaming=True,
    data_files="hf://datasets/Lichess/standard-chess-games@de4e636eddf568a9394cc01fb0b9e1da04a6babf/data/year=2025/month=09/train-00017-of-00062.parquet",
)

In [183]:
# Filter for games with evaluation
ds = ds.filter(lambda x: "[%eval" in x["movetext"])
# Randomize
ds = ds.shuffle(seed=1024, buffer_size=10000)

# Take 100000 games
N_games = 100000
games = list(ds.take(N_games))

In [184]:
print(len(games))
print(games[0])
print(type(games),type(games[0]))


100000
{'Event': 'Rated Bullet game', 'Site': 'https://lichess.org/RSQcYuc2', 'White': 'Pryb', 'Black': 'anya-forger', 'Result': '1-0', 'WhiteTitle': None, 'BlackTitle': None, 'WhiteElo': 1515, 'BlackElo': 1515, 'WhiteRatingDiff': 6, 'BlackRatingDiff': -5, 'UTCDate': datetime.date(2025, 9, 9), 'UTCTime': datetime.time(8, 13, 48), 'ECO': 'A16', 'Opening': 'English Opening: Anglo-Grünfeld Defense', 'Termination': 'Time forfeit', 'TimeControl': '60+0', 'movetext': '1. c4 { [%eval 0.12] [%clk 0:01:00] } 1... Nf6 { [%eval 0.14] [%clk 0:01:00] } 2. Nc3 { [%eval 0.12] [%clk 0:00:58] } 2... d5 { [%eval 0.26] [%clk 0:01:00] } 3. d4 { [%eval 0.07] [%clk 0:00:57] } 3... Bg4?! { [%eval 0.95] [%clk 0:00:59] } 4. Qb3 { [%eval 0.92] [%clk 0:00:55] } 4... Bc8?! { [%eval 1.7] [%clk 0:00:55] } 5. Nxd5?! { [%eval 0.83] [%clk 0:00:53] } 5... Nxd5 { [%eval 0.87] [%clk 0:00:52] } 6. cxd5 { [%eval 1.27] [%clk 0:00:52] } 6... b6? { [%eval 2.78] [%clk 0:00:46] } 7. e4 { [%eval 2.6] [%clk 0:00:50] } 7... g6 { [

In [224]:

# Convert to DataFrame
df = pd.DataFrame(games) 

In [225]:
df = df.drop(['Site',"WhiteTitle","BlackTitle","UTCDate","UTCTime"], axis=1)

In [ ]:
# We will only pick games with n>1000 instances. 
events_to_keep = ['Rated Blitz game', 'Rated Rapid game', 'Rated Bullet game','Rated Classical game']
df = df[df['Event'].isin(events_to_keep)].copy()
df['GameLength'] = df['movetext'].str.count(r'\b\d+\.\s')
df['Event'].value_counts()
# df.to_pickle('ChessGames.pkl')

In [229]:
small = df.sample(5000,random_state=42)

We will now create a new dataset where the actual moves made are stored and analyzed for blunders.

In [237]:
def extract_move_data(pgn_string, blunder_threshold=200):
    """
    Parses a PGN string with eval and clock tags into move-by-move data.
    """
    # python-chess needs standard PGN format to read properly
    game = chess.pgn.read_game(io.StringIO(pgn_string))
    if game is None:
        return []

    board = game.board()
    move_data = []
    
    # Starting position evaluation is roughly 0.20 pawns (20 centipawns) for White
    prev_eval_cp = 20 

    for node in game.mainline():
        # 1. State BEFORE the move
        is_white_turn = board.turn
        fen_before_move = board.fen()
        
        # 2. Extract Eval AFTER the move
        eval_obj = node.eval()
        if eval_obj is not None:
            # .white() gets the absolute score, and mate_score converts Mates to +/- 10000
            current_eval_cp = eval_obj.white().score(mate_score=10000)
        else:
            current_eval_cp = prev_eval_cp # Fallback if eval is missing
            
        # 3. Calculate Eval Difference (from the perspective of the player moving)
        if is_white_turn:
            # White wants the eval to be positive. A drop means prev > current.
            eval_drop = prev_eval_cp - current_eval_cp
        else:
            # Black wants the eval to be negative. A drop means current > prev.
            eval_drop = current_eval_cp - prev_eval_cp
            
        # 4. Label the Blunder
        is_blunder = 1 if eval_drop >= blunder_threshold else 0
        
        # 5. Extract Clock Time (in seconds)
        clock_time = node.clock() if node.clock() is not None else -1
        
        # 6. Append the row
        move_data.append({
            'FEN': fen_before_move,
            'Is_White_Turn': int(is_white_turn),
            'Move_Number': board.fullmove_number,
            'Clock_Seconds': clock_time,
            'Eval_Before': prev_eval_cp,
            'Eval_After': current_eval_cp,
            'Eval_Drop': eval_drop,
            'Target_Is_Blunder': is_blunder
        })
        
        # Push the move to update the board and update prev_eval
        board.push(node.move)
        prev_eval_cp = current_eval_cp
        
    return move_data

In [238]:
small['Move_Details'] = small['movetext'].apply(lambda x: extract_move_data(x, blunder_threshold=200))

In [241]:
df_exploded = small.explode('Move_Details').reset_index(drop=True)

In [250]:
move_df = pd.json_normalize(df_exploded['Move_Details'])
final_ml_df = pd.concat([df_exploded.drop(columns=['Move_Details', 'movetext']), move_df], axis=1)
final_ml_df = final_ml_df.dropna(subset=['FEN'])

In [255]:
final_ml_df.columns

Index(['Event', 'White', 'Black', 'Result', 'WhiteElo', 'BlackElo',
       'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'Termination',
       'TimeControl', 'GameLength', 'FEN', 'Is_White_Turn', 'Move_Number',
       'Clock_Seconds', 'Eval_Before', 'Eval_After', 'Eval_Drop',
       'Target_Is_Blunder'],
      dtype='object')

In [257]:
final_ml_df = final_ml_df.drop(columns=["FEN"])

In [ ]:
final_ml_df.to_pickle('ChessGamesWithMoves.pkl')

# 2. Processing Games

In [260]:
# g = games[0]
# print(g["movetext"])
# from io import StringIO
# # Sample game analysis
# game = pgn.read_game(StringIO(g["movetext"]))

# node = game

# while node.variations:
#     node = node.variation(0)
#     turn = node.parent.board().turn # Who just made that move
#     mateScore = 100000 # Score for mate

#     print(node.move) # The move
#     print(node.eval().white().score(mate_score=mateScore))   # Position eval in centipawns
#     print(node.eval().wdl(model="sf18", ply=node.ply()).pov(turn).expectation()) # Expected score
#     print(node.clock())  # seconds remaining
#     print('------------------')